# Multimodal Analysis of PCC-mPFC Connectivity Symptom Severity, and DRD4 Expression in ADHD

## Team Name: Patch Clamp Champs
- Enrique Aranda | A17484190   : 
- Brandon Melendez-Rodriguez | A17433726 : 

## Abstract

### Overview

This project will be analysing publicly available multimodal datasets to investigate whether resting state functional connectivity between two core Default Mode Network (DMN) regions, the posterior cingulate cortex (PCC) and medial prefrontal cortex (mPFC), differs between ADHD and control participants, and whether these differences relate to symptom severity and regional DRD4 expression. We used resting state fMRI and phenotype data from the ADHD-200 Consortium dataset to measure the PCC-mPFC connectivity and DSM-based symptom severity, and we used the Allen HUman Brain Atlas to estimate regional DRD4 gene expression as a molecular reference. This approach is important because it allows us to connect brain connectivity, behavior, and gene expression information in a single framework to examine ADHD.  




### Research Question

How does resting state functional connectivity of the Default Mode Network, specifically between the Posterior Cingulate Cortex and Medial Prefrontal Cortex, differ between ADHD and control participants, how is this connectivity related to DSM-based symptom severity, and how can regional DRD4 expression interpret these differences? 

## Introduction & Background 

### Prior Work 

ADHD is a neurodevelopmental disorder that is associated with inattention, impulsivity, and hyperactivity, and research suggests that these symptoms are linked to altered communication across large scale brain networks rather than dysfunction in a single brain region. *De La Fuente et al.* describes ADHD as involving structural and functional abnormalities in multiple neural systems, specifically networks related to attention, executive function, and resting state processing. One network that has been repeatedly implicated is the Default Mode Network (DMN), which includes the posterior cingulate cortex (PCC) and medial prefrontal cortex (mPFC).

In addition to altered brain network organization, dopamine related signaling has also been widely implicated in ADHD. *Volkow et al.* explains that dopamine genes, including DRD4, are part of the broader dopamine hypothesis of ADHD and are relevant to understanding reward, motivation, and attention related symptoms. DRD4 provides a molecular context for interpreting connectivity differences observed in the imaging data. Together, prior work supports a multimodal approach that combines brain connectivity, clinical symptoms, and dopamine related gene expression context to understand ADHD. 

### Dataset 

This project uses two public data sources to integrate three modalities. The first is the ADHD-200 dataset, which contains resting state fMRI, anatomical image, and phenotype data from children and adolescents with ADHD and control participants. (The ADHD-200 Consortium). We will be using the resting state fMRI data to measure functional connectivity between the posterior cingulate cortex (PCC) and medial prefrontal cortex (mPFC), and we use the phenotype data to measure DSM-based symptom severity. THe ADHD-200 Consortium highlights that this dataset is valuable because it combines imaging and phenotypic information in a large shared resource, making it possible to connect neural differences to behavioral measures.

The second source is the Allen Human Brain Atlas, which contains regional gene expression measurements across the human brain(Allen Insitute for Brain Science). From this dataset, we extract regional DRD4 expression near the PCC and mPFC coordinates used in the connectivity analysis. This gives us the third modality for the project, molecular context. Together, these datasets allow us to study resting state functional connectivity, symptom severity, and regional gene expression information in the same project. This multimodal design is important because it moves beyond asking whether ADHD and control groups differ in connectivity alone and instead asks how those differences relate to both behavior and dopamine related molecular context. 

### Hypothesis

We hypothesize that ADHD participants will show altered resting state functional connectivity between the posterior cingulate cortex (PCC) and medial prefrontal cortex (mPFC) compared with control participants. We further predict that more atypical PCC-mPFC connectivity will be associated with greater DSM based symptom severity. Finally, we expect that both PCC and mPFC will show measurable regional DRD4 expression, allowing dopamine related molecular context to help interpret any observed connectivity differences. Overall, our hypothesis is that integrating connectivity, behavior, and regional gene expression information will provide a clearer picture of ADHD related brain differences than any one modality alone . 

## Data Analysis

### Data Wrangling 

### Data Visualization

### Data Analysis & Results

#### Setup

In [1]:
# install if you dont have this
!pip install nilearn
# !pip install seaborn
# !pip install scipy

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ttest_ind, pearsonr

#ADHD-200 Preprocessed Dataset
from nilearn import datasets
from nilearn.maskers import NiftiSpheresMasker

Defaulting to user installation because normal site-packages is not writeable


/home/bmelendezrodriguez/.local/lib/python3.11/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.9.0' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [2]:
# Fetch ADHD-200 Dataset and save it to adhd_data
# https://nilearn.github.io/dev/modules/generated/nilearn.datasets.fetch_adhd.html

adhd_data = datasets.fetch_adhd(n_subjects=40)

[fetch_adhd] Dataset found in /home/bmelendezrodriguez/nilearn_data/adhd


In [3]:
# Download Allen Human Brian Atlas Dataset
# https://human.brain-map.org/api/v2/well_known_file_download/178238387

# metadata for human brain atlas data
probes_df = pd.read_csv('Probes.csv')
samples_df = pd.read_csv('SampleAnnot.csv')
samples_df.head()


FileNotFoundError: [Errno 2] No such file or directory: 'Probes.csv'

In [ ]:
pheno_df = pd.DataFrame(adhd_data.phenotypic)
# assigns subject as adhd or control
pheno_df['Group'] = pheno_df['adhd'].apply(
    lambda x: 'Control' if str(x).strip() == '0' else 'ADHD'
)

pheno_df.columns
pheno_df['dsm_iv_h_i'].value_counts()
pheno_df['dsm_iv_inatt'].value_counts()
pheno_df['dsm_iv_tot'].value_counts()

In [ ]:
# Internet-Assisted
# MNI coordinates are standardized coordinates for 3D neuroimaging
# the MNI coordinates for PCC is 0, -53, 26 and for mPFC is 0, 51, -6

pcc_coords  = (0, -53, 26)   # Posterior Cingulate Cortex
mpfc_coords = (0,  51,  -6)  # Medial Prefrontal Cortex

# Gemini-Assisted method, helps look at the brain at a specific area using the MNI coordinates
masker = NiftiSpheresMasker(
    seeds=[pcc_coords, mpfc_coords],
    radius=8.0,
    standardize='zscore_sample',
    detrend=True,
    low_pass=0.1,
    high_pass=0.01,
    t_r=2.0
)

connectivity_records = []
print('Extracting BOLD signals and computing PCC-mPFC connectivity...')


# the columns that have most information in the adhd_data
pheno_df['dsm_iv_inatt'] = pd.to_numeric(pheno_df['dsm_iv_inatt'], errors='coerce')
pheno_df['dsm_iv_h_i']   = pd.to_numeric(pheno_df['dsm_iv_h_i'], errors='coerce')
pheno_df['dsm_iv_tot']   = pd.to_numeric(pheno_df['dsm_iv_tot'], errors='coerce')
pheno_df['Calculated_Symptom_Score'] = pheno_df['dsm_iv_inatt'] + pheno_df['dsm_iv_h_i']
pheno_df['Calculated_Symptom_Score'] = pheno_df['Calculated_Symptom_Score'].fillna(pheno_df['dsm_iv_tot'])


for i in range( len(pheno_df)):
    func_file  = adhd_data.func[i]
    row        = pheno_df.iloc[i]

    # Extract time-series
    time_series  = masker.fit_transform(func_file)
    pcc_signal   = time_series[:, 0]
    mpfc_signal  = time_series[:, 1]

    # Pearson r then Fisher Z-transform for normality
    r_val, _ = pearsonr(pcc_signal, mpfc_signal)
    z_val    = np.arctanh(np.clip(r_val, -0.999, 0.999))

    symptom_score = row.get('Calculated_Symptom_Score', np.nan)
    connectivity_records.append({
        'Subject':        row['Subject'],
        'Group':          row['Group'],
        'Symptom_Score':  symptom_score,
        'Connectivity_Z': z_val
    })

results_df    = pd.DataFrame(connectivity_records)
results_df['Symptom_Score'] = pd.to_numeric(results_df['Symptom_Score'], errors='coerce')
results_clean = results_df.dropna(subset=['Symptom_Score'])

print(results_df['Group'].value_counts())

adhd_fc = results_df[results_df['Group'] == 'ADHD']['Connectivity_Z']
ctrl_fc = results_df[results_df['Group'] == 'Control']['Connectivity_Z']
t_stat, p_val_ttest = ttest_ind(adhd_fc, ctrl_fc, equal_var=False)


print(f'Processed {len(results_df)} subjects.')
print(f'T-Test (ADHD vs Control): t = {t_stat:.2f}, p = {p_val_ttest:.2f}')




The T-test is done to show differences between the funcitonal connectivity of ADHD patients vs Control patients. Since the p-value is greater than 0.05, the difference between the two groups is not statistically significant. Therefore, our data does not prove that the PCC and mPFC are more disconnected in the ADHD group vs the Control group.




In [ ]:
# Violin Plot showing functional connectivity
plt.figure(figsize=(6, 5))
sns.violinplot(x='Group', y='Connectivity_Z', data=results_df, palette=['#e74c3c', '#3498db'])

plt.title('DMN Functional Connectivity by Group')
plt.ylabel('Connectivity Strength')
plt.text(0.5, 0.92, f'p = {p_val_ttest:.2f}',
         ha='center', transform=plt.gca().transAxes,
         bbox=dict(facecolor='white', alpha=0.8))

plt.show()

This is a visualization of the t-test comparing the functional connectivities of the ADHD group and Control group. The violin plot shows us the density of the data with the width of the violin and shows us the spread of the data along the y-axis, which in this case is connectivity strength. Here we see more variety for the ADHD group as their strength varies from very strong to very weak with a spread out distribution. On the other hand, the Control group seems to normalize around average connectivity strength of 0.75 with hardly any diversity. Since the p-value is 0.28, which is over 0.05, the results are not statistically significant and the data does not prove there is a difference in functional connectivity.

In [ ]:
r_stat, p_val_corr = pearsonr(results_clean['Connectivity_Z'], results_clean['Symptom_Score'])
print(f'Correlation (Functional Connectivity vs Symptoms): r = {r_stat:.2f}, p = {p_val_corr:.2f}')

In addition, we measured the correlation between the functional connectivity of the Default Mode Network and the severity of the symptoms in patients, and we see a positive trend with a pearson's r-value of 0.92. Our p-value is slightly above 0.05 at 0.08, so our results are just barely statistically significant.

In [ ]:
plt.figure(figsize=(7, 6))
print(results_df)

sns.regplot(x='Connectivity_Z', y='Symptom_Score', data=results_df,
            scatter_kws={'alpha':0.5, 's':60})

plt.title('Brain-Behavior Relationship', fontsize=14, fontweight='bold')
plt.xlabel('PCC-mPFC Connectivity (Fisher Z)', fontsize=12)
plt.ylabel('Total DSM-IV Symptom Score', fontsize=12)
plt.text(0.05, 0.05, f'Pearson r = {r_stat:.3f}\np-value = {p_val_corr:.4f}',
          transform=plt.gca().transAxes, fontsize=11, fontweight='bold',
)
plt.savefig('brain_behavior_relationship.png', dpi=300, bbox_inches='tight')
plt.show()

Here, our

In [ ]:
drd4_probe_id = probes_df[probes_df['gene_symbol'] == 'DRD4']['probe_id'].iloc[0]

drd4_expression_values = None
print("Scanning MicroarrayExpression.csv for DRD4 (this may take a moment)...")

for chunk in pd.read_csv('MicroarrayExpression.csv', header=None, index_col=0, chunksize=1000):
    if drd4_probe_id in chunk.index:
        drd4_row = chunk.loc[drd4_probe_id]
        if isinstance(drd4_row, pd.DataFrame):
            drd4_row = drd4_row.iloc[0]
        drd4_expression_values = drd4_row.values
        break

if len(drd4_expression_values) > len(samples_df):
    drd4_expression_values = drd4_expression_values[:len(samples_df)]

def get_expression_within_radius(target_mni, samples_df, expression_values, radius_mm=15):
    sample_coords = samples_df[['mni_x', 'mni_y', 'mni_z']].values
    distances = np.linalg.norm(sample_coords - np.array(target_mni), axis=1)
    mask = distances <= radius_mm
    nearby_expression = expression_values[mask]

    if len(nearby_expression) == 0:
        return np.nan
    return np.mean(nearby_expression)

pcc_drd4  = get_expression_within_radius(pcc_coords, samples_df, drd4_expression_values, radius_mm=15)
mpfc_drd4 = get_expression_within_radius(mpfc_coords, samples_df, drd4_expression_values, radius_mm=15)

plt.figure(figsize=(7, 6))

df_genes = pd.DataFrame({
    'Region': ['PCC', 'mPFC'],
    'DRD4_Expression': [pcc_drd4, mpfc_drd4]
})

sns.barplot(x='Region', y='DRD4_Expression', data=df_genes,
            palette=['#95a5a6', '#f39c12'], edgecolor='black')

plt.title('Molecular Landscape: DRD4 Expression by DMN Hub', fontsize=14, fontweight='bold')
plt.ylabel('Microarray Expression Level (log2)', fontsize=12)
plt.xlabel('Region of Interest (ROI)', fontsize=12)

for index, row in df_genes.iterrows():
    plt.text(index, row['DRD4_Expression'] - 0.5, f"{row['DRD4_Expression']:.2f}",
             color='white', ha="center", fontweight='bold', fontsize=12)

plt.show()

## Conclusison & Discussion

### Results

### Limitations

### Future Work

## Reflection

## Contributions

| Tasks | Enrique Aranda | Brandon Melendez-Rodriguez |
|---|---|---|
| Ensure that everyone is completing their tasks | ✓ | ✓ |
| Identify research papers & question | ✓ | ✓ |
| Write background section |  | ✓ |
| Write code to wrangle datasets | ✓ | ✓ |
| Write code to analyze/integrate datasets | ✓ |  |
| Write code to visualize findings | ✓ | ✓ |
| Write markdown around code processes | ✓ |  |
| Write discussion section | ✓ | ✓ |
| Editing the text and code for grammar, misspellings, and clarity | ✓ | ✓ |

### References

- Allen Institute for Brain Science. https://human.brain-map.org/ 
- Attention Deficit Disorder Association (2022). Inside the ADHD Brain: Structure, Function, and Chemistry. https://add.org/adhd-brain/
- De La Fuente, A., et al. (2013). A review of attention-deficit/hyperactivity disorder from the perspective of brain networks. Frontiers in Human Neuroscience, 7. https://doi.org/10.3389/fnhum.2013.00192
- Milham, P. M., et al. (2012). The ADHD-200 Consortium: A model to advance the translational potential of neuroimaging in clinical neuroscience. Frontiers in Systems Neuroscience, (SEPTEMBER), 1-5. https://doi.org/10.3389/fnsys.2012.00062 
- Volkow, N. D., et al. (2009). Evaluating dopamine reward pathway in ADHD: clinical implications. JAMA, 302(10), 1084–1091. https://doi.org/10.1001/jama.2009.1308

